# 61 — Quỹ đầu tư: theo dấu dòng tiền tổ chức

`client.funds.*` là bề mặt mới của **finlens 1.4.0**: 188 quỹ Việt Nam, NAV
theo ngày từ 1995, danh mục nắm giữ theo tháng từ 2014.

Nhưng thứ đáng giá nhất không phải NAV — mà là **phép tra ngược**:

> `client.funds.holders("HPG")` → **80 quỹ đang nắm HPG**, kèm tỷ trọng và số
> lượng cổ phiếu.

Đây là cầu nối cổ phiếu ↔ tổ chức mà bốn nhóm dữ liệu cũ không có. Notebook
dựng một **bảng theo dấu dòng tiền tổ chức**: mã nào đang được quỹ gom, mã nào
đang bị xả, và quỹ nào đứng sau.

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, heatmap, hom_nay, lui_ngay, ty_dong

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

print(f"finlens {finlens.build_info()['version']} · pandas {pd.__version__}")
print(f"Năm phương thức: {[m for m in dir(client.funds) if not m.startswith('_')]}")

finlens 1.4.0 · pandas 3.0.5
Năm phương thức: ['holders', 'holdings', 'list', 'managers', 'nav']


## 1 · Vũ trụ 188 quỹ

`funds.list()` trả 28 cột, trong đó có bốn cột văn bản dài (`profile`,
`business_strategy`…) chứa HTML thô. Chọn cột trước khi in.

In [2]:
COT_GON = [
    "fund_code", "short_name", "fund_type", "fund_structure", "management_org",
    "total_nav", "nav_per_share_adjusted", "nav_point_count", "holding_period_count",
]

quy = client.funds.list()
print(f"{len(quy)} quỹ × {quy.shape[1]} cột")
print(f"Đơn vị: {{k: v for k, v in units if v}} = "
      f"{ {k: v for k, v in (quy.attrs['finlens']['units'] or {}).items() if v} }")
quy[COT_GON].head(5)

188 quỹ × 28 cột
Đơn vị: {k: v for k, v in units if v} = {'min_invest': 'VND', 'total_nav': 'VND', 'nav_per_share_adjusted': 'VND', 'foreign_ratio': 'ratio'}


,fund_code,short_name,fund_type,fund_structure,management_org,total_nav,nav_per_share_adjusted,nav_point_count,holding_period_count
0,A+ Fund,Quỹ Đầu tư A+,Quỹ thành viên,Quỹ cổ phiếu,Quản Lý Quỹ Đầu Tư FPT,NaN,NaN,0,0
1,ABBF,Quỹ Đầu tư TP An Bình,Quỹ mở,Quỹ trái phiếu,Quản lý quỹ An Bình,8.954411e+11,14495.79,1048,66
2,ABEF,Quỹ Đầu tư Cổ phiếu An Bình Thịnh Vượng,Quỹ mở,Quỹ cổ phiếu,Quản lý quỹ An Bình,3.828393e+10,8855.27,91,4
3,ABEIF,Quỹ Đầu tư Năng lượng và Hạ tầng An Bình,Quỹ thành viên,Quỹ cân bằng,Quản lý quỹ An Bình,NaN,NaN,0,0
4,ACB50,Quỹ Đầu tư ACB50,Quỹ thành viên,Quỹ cổ phiếu,Quản Lý Quỹ ACB,NaN,NaN,0,0


In [3]:
phan_loai = pd.crosstab(quy["fund_type"], quy["fund_structure"], margins=True, margins_name="Tổng")
print("Phân loại quỹ — `fund_type` là hình thức pháp lý, `fund_structure` là loại tài sản:")
phan_loai

Phân loại quỹ — `fund_type` là hình thức pháp lý, `fund_structure` là loại tài sản:


fund_structure,Quỹ BĐS,Quỹ cân bằng,Quỹ cổ phiếu,Quỹ trái phiếu,Tổng
fund_type,,,,,
Quỹ ETF,0,0,32,0,32
Quỹ mở,0,16,65,32,113
Quỹ thành viên,0,12,19,0,31
Quỹ đóng,1,1,10,0,12
Tổng,1,29,126,32,188


### ⚠️ Độ phủ có lỗ, và API **không** lọc chúng đi

Đây là điều đầu tiên phải kiểm trước khi tính bất cứ thống kê nào trên 188 quỹ.

In [4]:
do_phu = pd.DataFrame(
    {
        "số quỹ": [
            len(quy),
            (quy["nav_point_count"] == 0).sum(),
            (quy["holding_period_count"] == 0).sum(),
            ((quy["nav_point_count"] > 0) & (quy["holding_period_count"] > 0)).sum(),
        ]
    },
    index=["tổng danh mục", "chưa có dòng NAV nào", "chưa có danh mục nắm giữ", "có CẢ HAI"],
)
do_phu["tỷ lệ"] = (do_phu["số quỹ"] / len(quy) * 100).round(0).astype(int).astype(str) + "%"
do_phu

,số quỹ,tỷ lệ
tổng danh mục,188,100%
chưa có dòng NAV nào,39,21%
chưa có danh mục nắm giữ,47,25%
có CẢ HAI,141,75%


Một phân tích so hiệu suất toàn bộ 188 quỹ sẽ lặng lẽ bỏ qua 39 quỹ không có
dữ liệu — và tệ hơn, nếu bạn `fillna(0)` thì chúng thành những quỹ có NAV bằng 0.

**Lọc bằng `nav_point_count` trước.**

In [5]:
CO_NAV = quy[quy["nav_point_count"] > 0].copy()
CO_HOLDING = quy[quy["holding_period_count"] > 0].copy()
print(f"Quỹ dùng được cho phân tích NAV     : {len(CO_NAV)}")
print(f"Quỹ dùng được cho phân tích danh mục: {len(CO_HOLDING)}")

Quỹ dùng được cho phân tích NAV     : 149
Quỹ dùng được cho phân tích danh mục: 141


## 2 · ⚠️ Mã quỹ **không phải** mã chứng khoán

Hai khác biệt, cả hai đều làm gãy code viết theo thói quen mã cổ phiếu.

### 2.1 · Mã quỹ **không duy nhất**

Một mã có thể thuộc hai tổ chức khác nhau. Thư viện từ chối đoán và ném ngoại
lệ mang theo danh sách `organization_ids` để bạn chọn.

In [6]:
try:
    client.funds.nav("VVDIF")
except finlens.AmbiguousFundError as e:
    print(type(e).__name__)
    print(f"  {str(e)[:150]}…")
    IDS = e.organization_ids
    print()
    print(f"  organization_ids = {IDS}   (kiểu {type(IDS[0]).__name__})")

AmbiguousFundError
  [FL_VALIDATION_FUND_AMBIGUOUS] Mã quỹ 'VVDIF' thuộc 2 tổ chức. Mã quỹ không duy nhất, nên câu hỏi này không có đáp án duy nhất — hãy gọi lại bằng một …

  organization_ids = [8367518595247, 267411043169407]   (kiểu int)


### ⚠️ Ngoại lệ trả `int`, nhưng phương thức chỉ nhận chuỗi

Chi tiết nhỏ nhưng nó chặn bạn ngay ở dòng khắc phục — truyền thẳng phần tử của
`organization_ids` vào là `ValidationError`.

In [7]:
try:
    client.funds.holdings(IDS[0], on_error="raise")
except finlens.ValidationError as e:
    print(f"Truyền int → {type(e).__name__}: {str(e)[:80]}")

kq = client.funds.holdings(str(IDS[0]))
print(f"Truyền str → OK, {len(kq)} dòng")

Truyền int → ValidationError: [FL_VALIDATION] `fund` phải là chuỗi hoặc danh sách chuỗi, nhận được int. -> htt


Truyền str → OK, 1 dòng


### Chọn quỹ nào? Tra `funds.list()` rồi quyết định

`organization_ids` cho bạn hai lựa chọn nhưng không nói cái nào là cái bạn cần.
Bảng danh mục có tên đầy đủ và tổ chức quản lý — đủ để phân biệt.

In [8]:
trung_ma = quy[quy["fund_code"] == "VVDIF"][
    ["organization_id", "short_name", "management_org", "nav_point_count", "holding_period_count"]
]
print("Hai quỹ cùng mã VVDIF:")
print(trung_ma.to_string(index=False))

Hai quỹ cùng mã VVDIF:
 organization_id                             short_name               management_org  nav_point_count  holding_period_count
   8367518595247 Quỹ Đầu tư Khám Phá Giá trị Vietinbank       Quản Lý Quỹ Vietinbank                1                     1
 267411043169407   Quỹ Đầu tư Khám Phá Giá trị Việt Nam Quản lý Quỹ Trí Tuệ Việt Nam                0                     0


⚠️ Ở đây cả hai đều gần như **không có dữ liệu** — 1 và 0 điểm NAV. Đó là lý do
tôi không dùng VVDIF cho phần phân tích phía sau, và cũng là minh hoạ thứ hai
cho mục độ phủ ở trên: **mã tồn tại trong danh mục không đảm bảo có dữ liệu.**

### 2.2 · Mã quỹ giữ nguyên cách viết — **đừng viết hoa**

Mã cổ phiếu Việt Nam luôn là ba chữ in hoa, nên `.upper()` là phản xạ. Với quỹ
thì phản xạ đó làm hỏng lời gọi.

In [9]:
khong_chuan = quy[~quy["fund_code"].str.fullmatch(r"[A-Z0-9]+", na=False)]
print(f"{len(khong_chuan)}/{len(quy)} mã quỹ KHÔNG phải chỉ gồm [A-Z0-9]:")
print(f"  {khong_chuan['fund_code'].head(10).tolist()}")
print()
print("Chúng chứa dấu cách, dấu +, dấu -, hoặc chữ thường.")
print()

# Và có mã chứa cả ký tự KHÔNG NHÌN THẤY được
an = quy[quy["fund_code"].str.contains(chr(0xA0), na=False)]
if len(an):
    m = an["fund_code"].iloc[0]
    print(f"Mã có ký tự ẩn: {m!r}")
    print(f"  hiển thị ra trông như: {m}")
    print(f"  nhưng thật ra là     : {[hex(ord(c)) for c in m]}")
    print(f"  0xa0 = non-breaking space, KHÔNG phải dấu cách thường (0x20)")
    print()
    print(f"  Gõ tay dấu cách thường → {m.replace(chr(0xA0), ' ')!r} → không khớp mã nào")

53/188 mã quỹ KHÔNG phải chỉ gồm [A-Z0-9]:
  ['A+ Fund', 'ACBC-AGF', 'ACBC-BGF', 'ASP-VIET-A', 'ASP-VIET-SSF', 'ASP-VIETEQ-AI', 'ASP-VIETRMF', 'B-VIETNAM', 'CGS Fullgoal', 'CSOP FTSE VN']

Chúng chứa dấu cách, dấu +, dấu -, hoặc chữ thường.

Mã có ký tự ẩn: 'ETF\xa0FM'
  hiển thị ra trông như: ETF FM
  nhưng thật ra là     : ['0x45', '0x54', '0x46', '0xa0', '0x46', '0x4d']
  0xa0 = non-breaking space, KHÔNG phải dấu cách thường (0x20)

  Gõ tay dấu cách thường → 'ETF FM' → không khớp mã nào


In [10]:
MA_LA = "A+ Fund"
print(f"Gõ đúng như list() trả về: {MA_LA!r}")
try:
    client.funds.nav(MA_LA.upper(), on_error="raise")
except finlens.FinLensError as e:
    print(f"  .upper() → {type(e).__name__}: {str(e)[:80]}")

Gõ đúng như list() trả về: 'A+ Fund'


**Quy tắc: lấy mã từ chính `funds.list()`, đừng gõ tay và đừng chuẩn hoá.**

## 3 · NAV và hiệu suất quỹ

`funds.nav()` tự phân trang bằng cursor — bảng nguồn hơn 136.000 dòng, vượt xa
hạn mức một request.

In [11]:
import time

# Mười quỹ ETF lớn nhất theo NAV
ETF = (
    CO_NAV[CO_NAV["fund_type"] == "Quỹ ETF"]
    .nlargest(10, "total_nav")["fund_code"]
    .tolist()
)
print(f"Mười quỹ ETF lớn nhất: {ETF}")

t0 = time.perf_counter()
nav = client.funds.nav(ETF, start=lui_ngay(HOM_NAY, nam=2))
print(f"\n{len(nav):,} dòng NAV trong {time.perf_counter() - t0:.1f} giây")
print(f"truncated = {nav.attrs['finlens']['truncated']}")

Mười quỹ ETF lớn nhất: ['VNM-ETF', 'Fubon FTSE VN', 'ETF\xa0FM', 'FUEVFVND', 'Xtrackers Vietnam Swap UCITS ETF', 'E1VFVN30', 'KIM ACE VN30', 'FUEKIV30', 'FUEVN100', 'FUEMAV30']



4,629 dòng NAV trong 0.9 giây
truncated = False


⚠️ **Cờ `truncated` này từng sai ở bản 1.3.0.** Phân trang lấy trọn dữ liệu rồi
vẫn báo `True` kèm cảnh báo "Kết quả đã bị cắt", vì cờ của từng *trang* bị gộp
bằng `any(...)` thành khẳng định về cả *bảng*. Bản 1.4.0 sửa: cờ chỉ bật khi
phân trang **dừng sớm**, tức khi bảng thật sự thiếu.

### ⚠️ Đơn vị: VND thô, và bốn cột `_ratio` ở thang 0–1

In [12]:
don_vi = {k: v for k, v in nav.attrs["finlens"]["units"].items() if v}
print("Đơn vị các cột NAV:")
for c, u in don_vi.items():
    print(f"  {c:<24} {u}")

print()
print(f"nav_per_share trung bình: {nav['nav_per_share'].mean():,.0f} VND")
print("→ VND THÔ, khác kVND của giá cổ phiếu. Nhân 1.000 vào đây là sai 1.000 lần.")
print()
print(f"nav_change_ratio khoảng: {nav['nav_change_ratio'].min():.4f} … {nav['nav_change_ratio'].max():.4f}")
print("→ thang 0–1, KHÔNG phải phần trăm. Nhân 100 để hiển thị.")

Đơn vị các cột NAV:
  share_outstanding        share
  nav_per_share            VND
  nav_per_share_adjusted   VND
  nav_change_ratio         ratio
  total_nav                VND
  fund_flow                VND
  foreign_volume           share
  foreign_ratio            ratio
  close_price_adjusted     VND
  discount                 VND
  discount_ratio           ratio

nav_per_share trung bình: 215,750 VND
→ VND THÔ, khác kVND của giá cổ phiếu. Nhân 1.000 vào đây là sai 1.000 lần.

nav_change_ratio khoảng: -0.1333 … 0.1218
→ thang 0–1, KHÔNG phải phần trăm. Nhân 100 để hiển thị.


### So hiệu suất quỹ với VNINDEX

Dùng `nav_per_share_adjusted` — đã điều chỉnh cho phân phối, giống `adjusted`
của giá cổ phiếu.

In [13]:
vni = client.eod.index.ohlcv("VNINDEX", start=lui_ngay(HOM_NAY, nam=2)).sort_values("date")

def goc_100(df: pd.DataFrame, khoa: str, gia: str) -> pd.DataFrame:
    df = df.sort_values([khoa, "date"])
    dau = df.groupby(khoa, observed=True)[gia].transform("first")
    return df.assign(chi_so=df[gia] / dau * 100)


hieu_suat = goc_100(nav.dropna(subset=["nav_per_share_adjusted"]), "fund_code", "nav_per_share_adjusted")
chuan = vni.assign(chi_so=vni["close"] / vni["close"].iloc[0] * 100, fund_code="VNINDEX")

duong(
    pd.concat([hieu_suat[["date", "chi_so", "fund_code"]], chuan[["date", "chi_so", "fund_code"]]]),
    x="date",
    y="chi_so",
    theo="fund_code",
    tieu_de="Hiệu suất 10 quỹ ETF lớn nhất so với VNINDEX — 2 năm",
    phu_de="NAV/chứng chỉ đã điều chỉnh, chuẩn hoá về 100 tại phiên đầu",
    nhan_y="chỉ số (gốc = 100)",
)

In [14]:
tong_ket = (
    hieu_suat.groupby("fund_code", observed=True)
    .agg(phien=("date", "count"), chi_so_cuoi=("chi_so", "last"))
    .assign(thay_doi_pct=lambda d: (d["chi_so_cuoi"] - 100).round(2))
    .drop(columns="chi_so_cuoi")
    .sort_values("thay_doi_pct", ascending=False)
)
tong_ket.loc["VNINDEX"] = [len(chuan), round(chuan["chi_so"].iloc[-1] - 100, 2)]
print("Thay đổi 2 năm:")
print(tong_ket.to_string())

Thay đổi 2 năm:
                                  phien  thay_doi_pct
fund_code                                            
Xtrackers Vietnam Swap UCITS ETF  510.0         65.34
Fubon FTSE VN                     469.0         65.24
FUEKIV30                          514.0         54.18
E1VFVN30                          509.0         53.87
FUEMAV30                          517.0         53.66
VNM-ETF                           509.0         53.55
FUEVN100                          510.0         46.94
KIM ACE VN30                      479.0         40.89
FUEVFVND                          510.0          3.47
ETF FM                            102.0         -1.07
VNINDEX                           499.0         45.75


### ⚠️ `discount_ratio` là `null` với quỹ **không niêm yết**, không phải `0`

Chỉ một phần nhỏ quỹ có chứng chỉ giao dịch trên sàn. Với các quỹ còn lại,
"chênh lệch thị giá so với NAV" là một khái niệm **không tồn tại**.

In [15]:
co_thi_gia = nav.groupby("fund_code", observed=True)["close_price_adjusted"].apply(lambda s: s.notna().any())
print(f"Quỹ có thị giá: {co_thi_gia.sum()}/{len(co_thi_gia)} trong nhóm ETF đang xét")
print(f"Ô discount_ratio là NaN: {nav['discount_ratio'].isna().sum():,}/{len(nav):,}")

print()
print("Nếu nguồn để 0 thay vì NaN thì phép này sẽ sai:")
gia_su_0 = nav["discount_ratio"].fillna(0)
print(f"  mean() trên dữ liệu thật (NaN bị bỏ qua): {nav['discount_ratio'].mean():+.5f}")
print(f"  mean() nếu NaN thành 0                  : {gia_su_0.mean():+.5f}")
print(f"  → lệch {abs(nav['discount_ratio'].mean() - gia_su_0.mean()) / abs(nav['discount_ratio'].mean()):.0%}")

Quỹ có thị giá: 8/10 trong nhóm ETF đang xét
Ô discount_ratio là NaN: 662/4,629

Nếu nguồn để 0 thay vì NaN thì phép này sẽ sai:
  mean() trên dữ liệu thật (NaN bị bỏ qua): +0.00078
  mean() nếu NaN thành 0                  : +0.00067
  → lệch 14%


Đây là lý do thư viện đổi `0` của nguồn thành `null`: một mức chiết khấu bằng
đúng 0 là bất khả về mặt thị trường, nên số 0 ở nguồn có nghĩa là *vắng mặt*.

In [16]:
niem_yet = nav.dropna(subset=["discount_ratio"])
if len(niem_yet):
    chiet_khau = (
        niem_yet[niem_yet["date"] == niem_yet["date"].max()]
        .assign(chiet_khau_pct=lambda d: d["discount_ratio"] * 100)
        .sort_values("chiet_khau_pct")
    )
    bar_ngang(
        chiet_khau,
        nhan="fund_code",
        gia_tri="chiet_khau_pct",
        tieu_de=f"Chênh lệch thị giá so với NAV — phiên {niem_yet['date'].max():%d/%m/%Y}",
        phu_de="Dương = thị giá cao hơn NAV (premium) · âm = thấp hơn (discount)",
        nhan_x="%",
        dinh_dang_nhan="{:+.2f}%",
    )

## 4 · Danh mục nắm giữ của một quỹ

`funds.holdings()` nhận **một** quỹ và **một** kỳ. Không có `date` thì lấy kỳ
gần nhất của chính quỹ đó.

In [17]:
QUY = "VINACAPITAL-VESAF"
dm = client.funds.holdings(QUY)

print(f"{QUY} · kỳ {dm['date'].iloc[0]:%d/%m/%Y} · {len(dm)} khoản")
print(f"Loại tài sản: {dm['asset_type'].value_counts().to_dict()}")
print(f"Tổng tỷ trọng: {dm['weight_ratio'].sum():.2%}")
dm.nlargest(10, "weight_ratio")[["asset_code", "asset_name", "weight_ratio", "quantity"]].assign(
    ty_trong_pct=lambda d: (d["weight_ratio"] * 100).round(2)
).drop(columns="weight_ratio")

VINACAPITAL-VESAF · kỳ 30/06/2026 · 28 khoản
Loại tài sản: {'stock': 28}
Tổng tỷ trọng: 86.69%


,asset_code,asset_name,quantity,ty_trong_pct
1,BVH,Tập đoàn Bảo Việt,2815910.0,7.32
8,HPG,Hòa Phát,6639114.0,6.22
12,MBB,MBBank,5846380.0,5.92
25,VCB,Vietcombank,2364190.0,5.91
2,CTG,VietinBank,4273770.0,5.84
17,PNJ,Vàng Phú Nhuận,2204125.0,5.58
20,REE,Cơ Điện Lạnh REE,1874523.0,3.74
3,CTR,Công trình Viettel,1092310.0,3.73
14,MWG,Thế giới di động,1104160.0,3.47
7,GMD,Tập đoàn Gemadept,1071980.0,3.17


⚠️ **Tổng tỷ trọng không bằng 100%.** Phần còn lại là tiền mặt, tiền gửi và các
khoản không được liệt kê từng dòng. Đừng chuẩn hoá lại về 100% — làm thế là
giả định quỹ đầu tư toàn bộ, và thổi phồng mọi tỷ trọng.

In [18]:
print(f"Tổng tỷ trọng liệt kê : {dm['weight_ratio'].sum():.2%}")
print(f"Phần không liệt kê     : {1 - dm['weight_ratio'].sum():.2%}  ← tiền mặt và tài sản khác")
print()
sai = dm["weight_ratio"] / dm["weight_ratio"].sum()
lon_nhat = dm.nlargest(1, "weight_ratio").iloc[0]
print(f"Khoản lớn nhất ({lon_nhat['asset_code']}):")
print(f"  tỷ trọng thật        : {lon_nhat['weight_ratio']:.2%}")
print(f"  nếu chuẩn hoá về 100%: {sai.max():.2%}   ← thổi phồng")

Tổng tỷ trọng liệt kê : 86.69%
Phần không liệt kê     : 13.31%  ← tiền mặt và tài sản khác

Khoản lớn nhất (BVH):
  tỷ trọng thật        : 7.32%
  nếu chuẩn hoá về 100%: 8.44%   ← thổi phồng


### Danh mục đổi thế nào qua các kỳ

`holdings` nhận `date=`, nên so nhiều kỳ là một vòng lặp.

In [19]:
KY = ["2025-09-30", "2025-12-31", "2026-03-31", "2026-06-30"]

lich_su = pd.concat(
    [client.funds.holdings(QUY, date=k) for k in KY], ignore_index=True
)
print(f"{len(lich_su)} dòng qua {lich_su['date'].nunique()} kỳ")

bang_ky = (
    lich_su.assign(ky=lambda d: d["date"].dt.strftime("%m/%Y"), pct=lambda d: d["weight_ratio"] * 100)
    .pivot_table(index="asset_code", columns="ky", values="pct")
)
# Chỉ giữ các khoản từng chiếm ≥ 3% ở ít nhất một kỳ
bang_ky = bang_ky[bang_ky.max(axis=1) >= 3].sort_values(bang_ky.columns[-1], ascending=False)

heatmap(
    bang_ky.round(2),
    tieu_de=f"{QUY} — tỷ trọng các khoản nắm giữ qua bốn kỳ",
    phu_de="Chỉ các khoản từng chiếm ≥ 3% · ô trống là kỳ đó không nắm",
    nhan_mau="% danh mục",
    phan_ky=False,
    dinh_dang_o="%{z:.1f}",
)

115 dòng qua 4 kỳ


## 5 · Tra ngược — **ứng dụng chính**

`funds.holders()` trả lời câu hỏi ngược: *mã này đang được quỹ nào nắm?*

In [20]:
MA = "HPG"
nguoi_nam = client.funds.holders(MA)

print(f"{len(nguoi_nam)} quỹ đang nắm {MA}")
print(f"Tổng số cổ phiếu các quỹ nắm: {nguoi_nam['quantity'].sum():,.0f}")
nguoi_nam.nlargest(10, "quantity")[["fund_code", "fund_short_name", "date", "weight_ratio", "quantity"]].assign(
    ty_trong_pct=lambda d: (d["weight_ratio"] * 100).round(2)
).drop(columns="weight_ratio")

80 quỹ đang nắm HPG
Tổng số cổ phiếu các quỹ nắm: 630,337,279


,fund_code,fund_short_name,date,quantity,ty_trong_pct
49,PYNELITE,PYN Elite Fund (non-ucits),2026-06-30,142126336.0,13.40
66,VEIL,Vietnam Enterprise Investments Ltd.,2026-06-30,69229830.0,3.80
57,Tianhong VN,Tianhong Vietnamese Market Equity Launched QDI...,2026-06-30,57492053.0,8.65
73,VINACAPITAL-VOF,Vinacapital Vietnam Opportunity Fund,2026-06-30,52648030.0,5.00
31,Fubon FTSE VN,Fubon FTSE Vietnam ETF,2026-06-30,33376894.0,7.85
79,Xtrackers Vietnam Swap UCITS ETF,Xtrackers Vietnam Swap UCITS ETF 1C,2026-03-31,30167548.0,8.96
78,VNM-ETF,VanEck Vietnam ETF,2026-06-30,29738598.0,4.68
38,LVF,Lumen Vietnam Fund,2026-07-14,25388758.0,5.63
14,E1VFVN30,Quỹ ETF DCVFMVN30,2026-06-30,23021618.0,8.66
67,VFMVSF,Quỹ Đầu tư CP Việt Nam Chọn lọc,2026-06-30,15004479.0,3.93


## ⚠️ Cạm bẫy lớn nhất của cả notebook: cột `date` **khác nhau theo từng quỹ**

Nhìn cột `date` ở bảng trên. Nó **không** phải một kỳ chung.

`holders()` trả về **vị thế công bố gần nhất của mỗi quỹ**, và các quỹ công bố
theo lịch khác nhau — quý, tháng, hoặc chẳng theo lịch nào. Không có tài liệu
nào của thư viện nói điều này.

In [21]:
print(f"{len(nguoi_nam)} quỹ · {nguoi_nam['date'].nunique()} kỳ báo cáo khác nhau trong CÙNG một lời gọi")
print()
print(nguoi_nam["date"].dt.strftime("%Y-%m-%d").value_counts().sort_index().to_string())

80 quỹ · 5 kỳ báo cáo khác nhau trong CÙNG một lời gọi

date
2020-04-30     1
2026-03-31     1
2026-04-30     1
2026-06-30    76
2026-07-14     1


### Và đây là hậu quả bằng tiền

Quỹ báo cáo lần cuối từ nhiều năm trước **vẫn nằm trong danh sách**, với vị thế
của năm đó, như thể đó là vị thế hôm nay.

In [22]:
moi_nhat = nguoi_nam["date"].max()
cu_nhat = nguoi_nam.nsmallest(3, "date")

print(f"Kỳ mới nhất trong bảng: {moi_nhat:%d/%m/%Y}")
print()
print("Ba quỹ có kỳ báo cáo cũ nhất:")
print(
    cu_nhat[["fund_code", "date", "quantity", "weight_ratio"]]
    .assign(cu_bao_nhieu_ngay=lambda d: (moi_nhat - d["date"]).dt.days)
    .to_string(index=False)
)

Kỳ mới nhất trong bảng: 14/07/2026

Ba quỹ có kỳ báo cáo cũ nhất:
                       fund_code       date   quantity  weight_ratio  cu_bao_nhieu_ngay
                        PXP VEEF 2020-04-30 10913178.0        0.1670               2266
Xtrackers Vietnam Swap UCITS ETF 2026-03-31 30167548.0        0.0896                105
                         JFVNOPP 2026-04-30  8947446.0        0.0590                 75


In [23]:
NGUONG_NGAY = 180  # vị thế cũ hơn nửa năm thì không còn là "đang nắm"

cu = nguoi_nam[nguoi_nam["date"] < moi_nhat - pd.Timedelta(days=NGUONG_NGAY)]
tuoi = nguoi_nam[nguoi_nam["date"] >= moi_nhat - pd.Timedelta(days=NGUONG_NGAY)]

print(f"Tổng CHƯA lọc  : {nguoi_nam['quantity'].sum():>14,.0f} cp  ({len(nguoi_nam)} quỹ)")
print(f"Tổng ĐÃ lọc    : {tuoi['quantity'].sum():>14,.0f} cp  ({len(tuoi)} quỹ)")
print(f"Phần từ vị thế cũ hơn {NGUONG_NGAY} ngày: {cu['quantity'].sum():>,.0f} cp "
      f"= {cu['quantity'].sum() / nguoi_nam['quantity'].sum():.1%} tổng")

Tổng CHƯA lọc  :    630,337,279 cp  (80 quỹ)
Tổng ĐÃ lọc    :    619,424,101 cp  (79 quỹ)
Phần từ vị thế cũ hơn 180 ngày: 10,913,178 cp = 1.7% tổng


**Chỉ một quỹ lỗi thời đã chiếm hơn một phần tám tổng sở hữu được báo cáo.** Nó
không sai — nguồn ghi đúng những gì quỹ đó công bố lần cuối — nhưng gọi con số
đó là "quỹ đang nắm bao nhiêu" thì sai.

**Quy tắc: luôn lọc theo tuổi của vị thế trước khi cộng.**

In [24]:
def holders_tuoi(ma: str, *, date=None, nguong_ngay: int = 180) -> pd.DataFrame:
    """`holders()` đã lọc bỏ các vị thế quá cũ để không cộng nhầm.

    ⚠️ Cột `date` của `holders()` là kỳ công bố **của riêng từng quỹ**, không phải
    một kỳ chung. Cộng thẳng `quantity` là trộn vị thế của nhiều thời điểm khác
    nhau — có khi cách nhau nhiều năm.
    """
    d = client.funds.holders(ma, date=date)
    if d.empty:
        return d
    moc = d["date"].max()
    return d[d["date"] >= moc - pd.Timedelta(days=nguong_ngay)].copy()


sach = holders_tuoi(MA)
print(f"holders_tuoi({MA!r}) → {len(sach)} quỹ, kỳ từ {sach['date'].min():%d/%m/%Y} tới {sach['date'].max():%d/%m/%Y}")

holders_tuoi('HPG') → 79 quỹ, kỳ từ 31/03/2026 tới 14/07/2026


Hai cách đọc bảng này, và chúng trả lời hai câu khác nhau:

- **`quantity` lớn** → quỹ đó nắm nhiều cổ phiếu nhất. Quan trọng với thanh khoản.
- **`weight_ratio` lớn** → mã này chiếm tỷ trọng lớn trong danh mục quỹ. Quan
  trọng với mức độ *cam kết* của quỹ đó.

In [25]:
print("Xếp theo tỷ trọng trong danh mục — quỹ nào 'đặt cược' nhiều nhất vào mã này:")
print(
    nguoi_nam.nlargest(8, "weight_ratio")[["fund_code", "weight_ratio", "quantity"]]
    .assign(ty_trong_pct=lambda d: (d["weight_ratio"] * 100).round(2))
    .drop(columns="weight_ratio")
    .to_string(index=False)
)

Xếp theo tỷ trọng trong danh mục — quỹ nào 'đặt cược' nhiều nhất vào mã này:
fund_code    quantity  ty_trong_pct
 PXP VEEF  10913178.0         16.70
 FUCTVGF5   1474000.0         16.36
 FUCTVGF3   1716000.0         15.56
 FUCTVGF4   1650000.0         15.25
 FUEMITEC    310420.0         14.68
     LPLF    300000.0         13.67
 PYNELITE 142126336.0         13.40
    TCRES    939923.0         12.82


### Sở hữu của quỹ so với vốn hoá

Con số tuyệt đối không nói lên gì cho tới khi so với quy mô doanh nghiệp.

In [26]:
von_hoa = client.financials.indicators(MA, codes=["market_cap"], period="quarterly", start_year=HOM_NAY.year)
gia_hien_tai = client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, ngay=10))["close"].iloc[-1]

gia_tri_quy = nguoi_nam["quantity"].sum() * gia_hien_tai * 1_000  # kVND → VND
vh = von_hoa["value"].iloc[0]

print(f"{MA}: giá {gia_hien_tai:,.2f} nghìn VND · vốn hoá {vh / 1e12:,.1f} nghìn tỷ")
print(f"Các quỹ nắm {sach['quantity'].sum():,.0f} cp ≈ {gia_tri_quy / 1e12:,.2f} nghìn tỷ")
print(f"→ khoảng {gia_tri_quy / vh:.2%} vốn hoá")
print()
print("(dùng bảng ĐÃ lọc tuổi — nếu không thì con số này gồm cả vị thế của sáu năm trước)")

HPG: giá 22.10 nghìn VND · vốn hoá 196.7 nghìn tỷ
Các quỹ nắm 619,424,101 cp ≈ 13.93 nghìn tỷ
→ khoảng 7.08% vốn hoá

(dùng bảng ĐÃ lọc tuổi — nếu không thì con số này gồm cả vị thế của sáu năm trước)


## 6 · Quỹ đang **gom** hay **xả** — so hai ảnh chụp

`holders(ma, date=k)` cho một **ảnh chụp**: *tính tới ngày `k`, mỗi quỹ đang
nắm bao nhiêu theo lần công bố gần nhất của nó*.

So hai ảnh chụp là so **những gì bạn biết được ở hai thời điểm** — đúng cái bạn
muốn khi theo dấu dòng tiền, vì đó cũng là thứ thị trường biết được.

In [27]:
KY_TRUOC, KY_SAU = "2025-12-31", "2026-06-30"

truoc = holders_tuoi(MA, date=KY_TRUOC).set_index("fund_code")["quantity"]
sau = holders_tuoi(MA, date=KY_SAU).set_index("fund_code")["quantity"]

print(f"Ảnh chụp {KY_TRUOC}: {len(truoc)} quỹ · {truoc.sum() / 1e6:,.1f} triệu cp")
print(f"Ảnh chụp {KY_SAU}: {len(sau)} quỹ · {sau.sum() / 1e6:,.1f} triệu cp")
print(f"Thay đổi: {(sau.sum() - truoc.sum()) / 1e6:+,.1f} triệu cp")

Ảnh chụp 2025-12-31: 70 quỹ · 464.4 triệu cp
Ảnh chụp 2026-06-30: 79 quỹ · 618.9 triệu cp
Thay đổi: +154.5 triệu cp


In [28]:
so_sanh = pd.DataFrame({"truoc": truoc, "sau": sau})
so_sanh["trang_thai"] = np.select(
    [so_sanh["truoc"].isna(), so_sanh["sau"].isna()],
    ["mới xuất hiện", "biến mất"],
    default="có ở cả hai",
)
print(so_sanh["trang_thai"].value_counts().to_string())

trang_thai
có ở cả hai      68
mới xuất hiện    11
biến mất          2


⚠️ **Ba nhóm này không được gộp chung.**

- *có ở cả hai* → chênh lệch là thay đổi vị thế thật
- *mới xuất hiện* → quỹ mới mua, **hoặc** quỹ cũ vừa công bố lần đầu sau lâu ngày
- *biến mất* → quỹ đã bán hết, **hoặc** vị thế của nó vừa quá hạn lọc tuổi

Chỉ nhóm đầu tiên đo được thay đổi. Hai nhóm sau lẫn hành vi giao dịch với độ
trễ công bố, và dữ liệu **không** phân biệt được hai thứ đó.

In [29]:
ca_hai = so_sanh[so_sanh["trang_thai"] == "có ở cả hai"].copy()
ca_hai["thay_doi"] = ca_hai["sau"] - ca_hai["truoc"]
ca_hai["trieu_cp"] = (ca_hai["thay_doi"] / 1e6).round(2)

dong_thuc = ca_hai[ca_hai["thay_doi"] != 0]
print(f"{len(ca_hai)} quỹ có mặt ở cả hai ảnh chụp")
print(f"  {len(dong_thuc)} quỹ có thay đổi vị thế")
giu_nguyen = len(ca_hai) - len(dong_thuc)
print(f"  {giu_nguyen} quỹ giữ nguyên số lượng")
if giu_nguyen:
    print("    ⚠️ có thể là không giao dịch, cũng có thể là chưa công bố lại nên hai")
    print("       ảnh chụp cùng đọc một bản công bố — dữ liệu không phân biệt được")

68 quỹ có mặt ở cả hai ảnh chụp
  68 quỹ có thay đổi vị thế
  0 quỹ giữ nguyên số lượng


In [30]:
ve = pd.concat([ca_hai.nlargest(7, "thay_doi"), ca_hai.nsmallest(7, "thay_doi")]).reset_index()
bar_ngang(
    ve,
    nhan="fund_code",
    gia_tri="trieu_cp",
    tieu_de=f"{MA} — quỹ gom và xả nhiều nhất, {KY_TRUOC} → {KY_SAU}",
    phu_de="Chỉ các quỹ có mặt ở CẢ HAI ảnh chụp — quỹ mới xuất hiện hoặc biến mất đã bị loại",
    nhan_x="triệu cổ phiếu",
    dinh_dang_nhan="{:+.2f}",
)

In [31]:
tuoi_vi_the = (moi_nhat - nguoi_nam["date"]).dt.days
print("Tuổi của vị thế được báo cáo, tính từ kỳ mới nhất trong bảng:")
print(
    pd.cut(tuoi_vi_the, bins=[-1, 30, 90, 180, 365, 99999],
           labels=["≤ 1 tháng", "1–3 tháng", "3–6 tháng", "6–12 tháng", "> 1 năm"])
    .value_counts()
    .sort_index()
    .to_string()
)

Tuổi của vị thế được báo cáo, tính từ kỳ mới nhất trong bảng:
date
≤ 1 tháng     77
1–3 tháng      1
3–6 tháng      1
6–12 tháng     0
> 1 năm        1


## 7 · Sản phẩm — bảng theo dấu dòng tiền tổ chức

Ghép tất cả: với một danh sách mã, mã nào đang được quỹ gom mạnh nhất — và chỉ
tính các quỹ **có mặt ở cả hai ảnh chụp**, để không lẫn độ trễ công bố vào.

In [32]:
DANH_SACH = ["HPG", "VCB", "FPT", "MWG", "VNM", "TCB", "MBB", "ACB", "SSI", "VIC"]


def theo_dau_mot_ma(ma: str, ky_truoc: str, ky_sau: str) -> dict | None:
    """So hai ảnh chụp sở hữu của quỹ với một mã, đã lọc tuổi vị thế."""
    a = holders_tuoi(ma, date=ky_truoc)
    b = holders_tuoi(ma, date=ky_sau)
    if b.empty:
        return None

    sa = a.set_index("fund_code")["quantity"] if not a.empty else pd.Series(dtype=float)
    sb = b.set_index("fund_code")["quantity"]
    chung = sa.index.intersection(sb.index)

    return {
        "mã": ma,
        "quỹ đầu kỳ": len(sa),
        "quỹ cuối kỳ": len(sb),
        "quỹ có ở cả hai": len(chung),
        "triệu cp cuối kỳ": round(sb.sum() / 1e6, 2),
        # Chỉ cộng thay đổi của các quỹ so được — bỏ quỹ mới xuất hiện/biến mất
        "thay đổi triệu cp": round((sb[chung].sum() - sa[chung].sum()) / 1e6, 2),
        "tỷ trọng bq %": round(b["weight_ratio"].mean() * 100, 2),
    }


dong = [r for ma in DANH_SACH if (r := theo_dau_mot_ma(ma, KY_TRUOC, KY_SAU))]
bang = pd.DataFrame(dong).sort_values("thay đổi triệu cp", ascending=False)

print(f"Theo dấu sở hữu của quỹ, {KY_TRUOC} → {KY_SAU}")
print("(cột 'thay đổi' chỉ tính các quỹ có mặt ở cả hai ảnh chụp)")
bang

Theo dấu sở hữu của quỹ, 2025-12-31 → 2026-06-30
(cột 'thay đổi' chỉ tính các quỹ có mặt ở cả hai ảnh chụp)


,mã,quỹ đầu kỳ,quỹ cuối kỳ,quỹ có ở cả hai,triệu cp cuối kỳ,thay đổi triệu cp,tỷ trọng bq %
1,VCB,50,62,45,117.78,12.48,4.35
4,VNM,39,47,33,46.13,1.93,2.76
8,SSI,40,47,34,103.25,1.11,2.71
2,FPT,68,64,51,115.99,-7.04,4.88
5,TCB,61,69,59,219.85,-16.59,5.02
6,MBB,68,74,66,304.14,-18.60,5.35
0,HPG,70,79,68,618.91,-20.42,6.24
3,MWG,66,77,65,193.48,-29.49,5.62
7,ACB,56,62,51,283.93,-32.76,4.68
9,VIC,41,49,38,71.87,-39.46,9.75


In [33]:
bar_ngang(
    bang,
    nhan="mã",
    gia_tri="thay đổi triệu cp",
    tieu_de=f"Quỹ gom hay xả — {KY_TRUOC} đến {KY_SAU}",
    phu_de="Chỉ tính các quỹ có mặt ở cả hai ảnh chụp",
    nhan_x="triệu cổ phiếu",
    dinh_dang_nhan="{:+.1f}",
)

### Đối chiếu với giá — dòng tiền quỹ có đi cùng giá không?

⚠️ Đây là **quan hệ đồng thời trên 10 quan sát**, không phải bằng chứng dự báo.
Danh mục quỹ công bố **sau** kỳ báo cáo, nên khi bạn đọc được nó thì giao dịch
đã xong từ lâu.

In [34]:
gia_ky = client.eod.stock.ohlcv(bang["mã"].tolist(), start=KY_TRUOC)
ls = (
    gia_ky.sort_values(["symbol", "date"])
    .groupby("symbol", observed=True)["close"]
    .agg(lambda s: (s.iloc[-1] / s.iloc[0] - 1) * 100)
    .rename("ls_ky_pct")
    .round(2)
)

doi_chieu = bang.set_index("mã").join(ls)
r = doi_chieu["thay đổi triệu cp"].corr(doi_chieu["ls_ky_pct"])
print(f"Tương quan giữa 'quỹ gom' và 'giá tăng': {r:+.3f}  (n={len(doi_chieu)})")
print(f"⚠️ n={len(doi_chieu)} thì sai số chuẩn của hệ số tương quan khoảng "
      f"{1 / (len(doi_chieu) - 3) ** 0.5:.2f} — con số trên KHÔNG phân biệt được với 0.")
print()
print(doi_chieu[["quỹ có ở cả hai", "thay đổi triệu cp", "triệu cp cuối kỳ", "ls_ky_pct"]].to_string())

Tương quan giữa 'quỹ gom' và 'giá tăng': -0.369  (n=10)
⚠️ n=10 thì sai số chuẩn của hệ số tương quan khoảng 0.38 — con số trên KHÔNG phân biệt được với 0.

     quỹ có ở cả hai  thay đổi triệu cp  triệu cp cuối kỳ  ls_ky_pct
mã                                                                  
VCB               45              12.48            117.78       4.68
VNM               33               1.93             46.13       4.29
SSI               34               1.11            103.25     -16.36
FPT               51              -7.04            115.99     -25.08
TCB               59             -16.59            219.85      -7.84
MBB               66             -18.60            304.14       0.94
HPG               68             -20.42            618.91      -6.24
MWG               65             -29.49            193.48     -14.98
ACB               51             -32.76            283.93      10.01
VIC               38             -39.46             71.87      27.06


## 8 · `managers()` — ai điều hành quỹ

In [35]:
nguoi = client.funds.managers("E1VFVN30")
print(f"{len(nguoi)} người:")
print(nguoi[["full_name", "position_name"]].to_string(index=False))

5 người:
          full_name               position_name
 Nguyễn Bội Hồng Lê   Chủ tịch Ban đại diện Quỹ
  Lương Thị Mỹ Hạnh         Người điều hành quỹ
         Vũ Đức Sửu         Người điều hành quỹ
   Lê Thị Thu Hương Thành viên Ban đại diện Quỹ
Phạm Thị Thanh Thúy Thành viên Ban đại diện Quỹ


## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Danh mục 188 quỹ | `funds.list(fund_type="Quỹ ETF")` |
| NAV theo ngày | `funds.nav(ma_quy, start=…)` — tự phân trang |
| Danh mục nắm giữ một quỹ | `funds.holdings(quy, date=…)` — một quỹ, một kỳ |
| **Mã này quỹ nào nắm** | `funds.holders("HPG", date=…)` |
| Người điều hành | `funds.managers(quy)` |

**Sáu điều mang theo:**

1. ⚠️ **Mã quỹ không duy nhất.** `AmbiguousFundError` mang `organization_ids` —
   bắt nó và gọi lại bằng id, nhưng nhớ `str()`: ngoại lệ trả `int` còn phương
   thức chỉ nhận chuỗi.
2. ⚠️ **Mã quỹ giữ nguyên cách viết.** 53/188 mã chứa dấu cách, dấu `+`, hoặc
   chữ thường. `.upper()` làm hỏng lời gọi. Lấy mã từ `funds.list()`.
3. ⚠️ **Tiền tệ là VND thô**, khác `kVND` của giá cổ phiếu. Bốn cột `_ratio` ở
   thang **0–1**, không phải phần trăm.
4. ⚠️ **`discount_ratio` là `null` với quỹ không niêm yết**, không phải `0` —
   một mức chiết khấu bằng đúng 0 là bất khả, nên số 0 ở nguồn nghĩa là vắng mặt.
5. ⚠️ **Tổng tỷ trọng danh mục không bằng 100%.** Phần còn lại là tiền mặt.
   Chuẩn hoá lại về 100% là thổi phồng mọi khoản.
6. ⚠️ **Quỹ vắng mặt ở một kỳ có hai nghĩa**: đã bán hết, hoặc chưa công bố. Dữ
   liệu không phân biệt được, nên đừng đọc toàn bộ chênh lệch thành giao dịch.

---

**Ba bề mặt khác cũng mới ở finlens 1.4.0** và chưa có notebook:
`client.bonds.*` (trái phiếu doanh nghiệp), `client.financials.notes()` (thuyết
minh BCTC tổ chức tín dụng), và năm cột mới của `meta.warrants()` — trong đó
`conversion_ratio` là mảnh còn thiếu để tính giá trị nội tại chứng quyền, thứ
notebook `43` từng phải bỏ qua.